# LaLonde Dataset: Direct Covariate Matching

This notebook demonstrates direct covariate matching (without propensity scores) on the LaLonde dataset to estimate the average treatment effect on the treated (ATT) of job training on earnings.

Unlike propensity score matching which reduces covariates to a single score, this approach matches treated and control units based on **Mahalanobis distance** in the full covariate space. Mahalanobis distance accounts for correlations between covariates and differences in variance, making it superior to Euclidean distance for matching.

We use **observational data** (experimental treated units + non-experimental PSID control group) which has selection bias, and compare our estimate against the true experimental ATT from the randomized trial.

In [1]:
# Add project root to path so we can import the datasets module
import sys
from pathlib import Path
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

In [2]:
import pandas as pd
import numpy as np
from datasets import LalondeDataset
from methods import MahalanobisMatch

In [3]:
# Load LaLonde dataset with observational data
# This combines experimental treated (185) with non-experimental PSID controls (2,490)
ds = LalondeDataset()
data = ds.experimental

print(f"Dataset shape: {data.shape}")
print(f"\nColumns: {list(data.columns)}")
print(f"\nFirst few rows:")
data.head()

Dataset shape: (445, 12)

Columns: ['treat', 'age', 'educ', 'black', 'hisp', 'married', 'nodegr', 're74', 're75', 're78', 'u74', 'u75']

First few rows:


,treat,age,educ,black,hisp,married,nodegr,re74,re75,re78,u74,u75
0,True,37.0,11.0,1.0,0.0,1.0,1.0,0.0,0.0,9930.0460,1.0,1.0
1,True,22.0,9.0,0.0,1.0,0.0,1.0,0.0,0.0,3595.8940,1.0,1.0
2,True,30.0,12.0,1.0,0.0,0.0,0.0,0.0,0.0,24909.4500,1.0,1.0
3,True,27.0,11.0,1.0,0.0,0.0,1.0,0.0,0.0,7506.1460,1.0,1.0
4,True,33.0,8.0,1.0,0.0,0.0,1.0,0.0,0.0,289.7899,1.0,1.0


In [4]:
# Define treatment, outcome, and covariates
treatment_col = "treat"
outcome_col = "re78"
covariate_cols = ["age", "educ", "black", "hisp", "married", "nodegr", "re74", "re75"]

In [21]:
# Perform nearest neighbor matching using Mahalanobis distance
matcher = MahalanobisMatch(replace=True, ratio=5, caliper=0.5)
matched_data = matcher.match(
    data=data,
    treatment_col=treatment_col,
    outcome_col=outcome_col,
    covariate_cols=covariate_cols
)

print(f"Original data: {len(data)} rows")
print(f"Matched data: {len(matched_data)} rows")
print(f"Treated units: {matched_data[treatment_col].sum()}")
print(f"Control units: {(~matched_data[treatment_col]).sum()}")
print(f"\nMean Mahalanobis distance: {matcher.distances_.mean():.2f}")
print(f"Median Mahalanobis distance: {np.median(matcher.distances_):.2f}")

Original data: 445 rows
Matched data: 744 rows
Treated units: 372
Control units: 372

Mean Mahalanobis distance: 0.17
Median Mahalanobis distance: 0.15


In [22]:
# Compute Average Treatment Effect on the Treated (ATT)
att = matcher.estimate_att()

treated_outcome = matched_data.loc[matched_data[treatment_col] == 1, outcome_col].mean()
control_outcome = matched_data.loc[matched_data[treatment_col] == 0, outcome_col].mean()

print(f"\nAverage outcome for treated: ${treated_outcome:,.2f}")
print(f"Average outcome for control: ${control_outcome:,.2f}")
print(f"\nEstimated ATT: ${att:,.2f}")


Average outcome for treated: $5,148.20
Average outcome for control: $3,316.96

Estimated ATT: $1,831.24


In [23]:
# Compare with true experimental effect
# The true ATT comes from the randomized experiment (NSW demonstration)
print(f"\n{'='*60}")
print(f"Comparison with Experimental Benchmark")
print(f"{'='*60}")
print(f"True experimental ATT:      ${ds.true_att:,.2f}")
print(f"Mahalanobis matching ATT:   ${att:,.2f}")
print(f"Difference:                 ${att - ds.true_att:,.2f} ({(att - ds.true_att) / ds.true_att * 100:.1f}%)")


Comparison with Experimental Benchmark
True experimental ATT:      $1,794.34
Mahalanobis matching ATT:   $1,831.24
Difference:                 $36.90 (2.1%)


In [24]:
# Check covariate balance after matching
print("\nCovariate Balance (Standardized Mean Differences):")
print("=" * 60)

for col in covariate_cols:
    treated_mean = matched_data.loc[matched_data[treatment_col] == 1, col].mean()
    control_mean = matched_data.loc[matched_data[treatment_col] == 0, col].mean()
    pooled_sd = np.sqrt(
        (matched_data.loc[matched_data[treatment_col] == 1, col].var() + 
         matched_data.loc[matched_data[treatment_col] == 0, col].var()) / 2
    )
    smd = (treated_mean - control_mean) / pooled_sd if pooled_sd > 0 else 0
    print(f"{col:12s}: {smd:7.4f}")


Covariate Balance (Standardized Mean Differences):
age         : -0.0010
educ        :  0.0000
black       :  0.0000
hisp        :  0.0000
married     :  0.0000
nodegr      :  0.0000
re74        :  0.1276
re75        :  0.0953
